# grapheme-aware String Distance 

A **grapheme-aware** string distance library that wraps [`textdistance`](https://github.com/life4/textdistance) with proper Unicode grapheme cluster handling via `grapheme_kit.Graphemizer`.

## Setup


In [1]:
# Install dependencies
# pip install textdistance grapheme_kit

from pathlib import Path
import sys

project_root = Path.cwd()
for candidate in (project_root, project_root.parent, project_root.parent.parent):
    if (candidate / 'src').exists():
        project_root = candidate
        break

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from grapheme_kit.distance import damerau_levenshtein, hamming, jaro, jaro_winkler, levenshtein, longest_common_subsequence


---
## Levenshtein Distance

**Operations:** insert, delete, substitute  
**Returns:** minimum number of single grapheme edits to transform `s1` into `s2`


In [2]:
# Example 1: Grapheme deletion
print("Deletion")
print(f'levenshtein("किताब", "कताब") = {levenshtein("किताब", "कताब")}')
print()

# Example 2: Grapheme insertion
print("Insertion")
print(f'levenshtein("कताब", "किताब") = {levenshtein("कताब", "किताब")}')
print()

# Example 3: Grapheme substitution
print("Substitution")
print(f'levenshtein("घर", "घर") = {levenshtein("घर", "घर")}')
print()

# Example 4: Identical strings
print("Identical strings")
print(f'levenshtein("सुप्रभात", "सुप्रभात") = {levenshtein("सुप्रभात", "सुप्रभात")}')

Deletion
levenshtein("किताब", "कताब") = 1

Insertion
levenshtein("कताब", "किताब") = 1

Substitution
levenshtein("घर", "घर") = 0

Identical strings
levenshtein("सुप्रभात", "सुप्रभात") = 0


---
## Hamming Distance

**Compares:** grapheme at position 1 with grapheme at position 1, position 2 with position 2, and so on.  
**Returns:** number of positions that differ; extra trailing graphemes are counted as differences.  
**Best for:** strings where position matters, such as fixed-width labels, codes, and aligned OCR output.

Hamming distance is useful when you do not want edits to shift the rest of the string. A substitution changes one position, but an insertion or deletion can make many later positions differ.


In [3]:
print("Identical Arabic graphemes")
print(f'hamming("كتاب", "كتاب") = {hamming("كتاب", "كتاب")}')
print()

print("Classic aligned Arabic examples")
print(f'hamming("سلام", "سلام") = {hamming("سلام", "سلام")}')
print(f'hamming("بيت", "بنت") = {hamming("بيت", "بنت")}')
print()

print("Unequal grapheme counts")
print(f'hamming("علم", "عالم") = {hamming("علم", "عالم")}')
print(f'hamming("مرحبا", "") = {hamming("مرحبا", "")}')

Identical Arabic graphemes
hamming("كتاب", "كتاب") = 0

Classic aligned Arabic examples
hamming("سلام", "سلام") = 0
hamming("بيت", "بنت") = 1

Unequal grapheme counts
hamming("علم", "عالم") = 3
hamming("مرحبا", "") = 5


---
## Damerau-Levenshtein Distance

**Operations:** insert, delete, substitute, transpose two adjacent graphemes.  
**Returns:** minimum number of grapheme++ edits, with adjacent swaps counted as one edit.  
**Best for:** typo-tolerant matching where swapped neighboring graphemes are common.

This is close to Levenshtein, but it gives a cheaper score to transposition errors such as `teh` instead of `the`.


In [4]:
pairs = [
    ("घर", "रघ"),
    ("किताब", "कतिबा"),
    ("नमस्ते", "नमस्कार"),
]

header = f"{'s1':<10} {'s2':<10} {'Lev':>4} {'D-L':>4}"
print(header)
print("-" * len(header))
for s1, s2 in pairs:
    print(f"{s1:<10} {s2:<10} {levenshtein(s1, s2):>4} {damerau_levenshtein(s1, s2):>4}")


s1         s2          Lev  D-L
-------------------------------
घर         रघ            2    1
किताब      कतिबा         3    3
नमस्ते     नमस्कार       2    2


---
## Jaro Similarity

**Range:** `0.0` means no similarity; `1.0` means identical.  
**Compares:** matching graphemes within a moving window, then penalizes transpositions.  
**Best for:** short strings such as names, identifiers, and record-linkage fields.

Jaro is a similarity score, not a distance. Higher is better.


In [5]:
pairs = [
    # Hindi (Devanagari)
    ("किताब", "कताब"),

    # Arabic
    ("كتاب", "كتابا"),

    # Chinese
    ("你好", "你号"),

    # Japanese
    ("こんにちは", "こんにちわ"),

    # Korean
    ("안녕하세요", "안녕하새요"),

    # Russian (Cyrillic)
    ("привет", "превет"),
]
print(f"{'s1':<14} {'s2':<14} {'Jaro':>6}")
print("-" * 36)
for s1, s2 in pairs:
    print(f"{s1:<14} {s2:<14} {jaro(s1, s2):>6.4f}")


s1             s2               Jaro
------------------------------------
किताब          कताब           0.7778
كتاب           كتابا          0.9333
你好             你号             0.6667
こんにちは          こんにちわ          0.8667
안녕하세요          안녕하새요          0.8667
привет         превет         0.8222


---
## Jaro-Winkler Similarity

**Range:** `0.0` to `1.0`, like Jaro.  
**Adds:** a prefix boost for strings that begin with the same graphemes.  
**Best for:** names, titles, and short fields where early graphemes are especially meaningful.

Use Jaro-Winkler when a shared beginning should make two strings feel closer than plain Jaro would score them.


In [6]:
pairs = [
    ("घर", "रघ"),
    ("किताब", "कतिबा"),
    ("नमस्ते", "नमस्कार"),
]
print(f"{'s1':<18} {'s2':<18} {'Jaro':>6} {'Jaro-W':>7}")
print("-" * 52)
for s1, s2 in pairs:
    print(f"{s1:<18} {s2:<18} {jaro(s1,s2):>6.4f} {jaro_winkler(s1,s2):>7.4f}")

print()
print("Jaro-Winkler is usually >= Jaro when the strings share a prefix.")


s1                 s2                   Jaro  Jaro-W
----------------------------------------------------
घर                 रघ                 0.0000  0.0000
किताब              कतिबा              0.0000  0.0000
नमस्ते             नमस्कार            0.7833  0.8483

Jaro-Winkler is usually >= Jaro when the strings share a prefix.


---
## Longest Common Subsequence (LCS)

**Returns:** length of the longest grapheme sequence that appears in both strings in the same order.  
**Allows:** gaps; the shared graphemes do not need to be next to each other.  
**Best for:** diff-style comparisons, partial overlap, and measuring preserved order.

LCS is about what the two strings keep in common, rather than the number of edits needed to transform one into the other.


In [7]:
print("किताब vs कताब:", longest_common_subsequence("किताब", "कताब"))
print("नमस्ते vs नमस्ते:", longest_common_subsequence("नमस्ते", "नमस्ते"))
print("घर vs पेड़:", longest_common_subsequence("घर", "पेड़"))

किताब vs कताब: 2
नमस्ते vs नमस्ते: 4
घर vs पेड़: 0


---
## Side-by-Side Comparison

The distance metrics get larger as strings become more different. The similarity metrics get closer to `1.0` as strings become more alike.


In [8]:
dataset = [
    # Hindi
    ("घर", "घर"),
    ("किताब", "कताब"),

    # Arabic
    ("كتاب", "كتبا"),

    # Chinese
    ("你好", "你号"),

    # Japanese
    ("こんにちは", "こんにちわ"),

    # Korean
    ("학교", "학꾜"),
]

header = f"{'s1':<12} {'s2':<12} {'Lev':>4} {'DL':>4} {'Jaro':>6} {'J-W':>6} {'LCS':>4}"
print(header)
print("-" * len(header))
for s1, s2 in test_pairs:
    lev = levenshtein(s1, s2)
    dl  = damerau_levenshtein(s1, s2)
    j   = jaro(s1, s2)
    jw  = jaro_winkler(s1, s2)
    lcs = longest_common_subsequence(s1, s2)
    print(f"{s1:<12} {s2:<12} {lev:>4} {dl:>4} {j:>6.3f} {jw:>6.3f} {lcs:>4}")


s1           s2            Lev   DL   Jaro    J-W  LCS
------------------------------------------------------


NameError: name 'test_pairs' is not defined

---
## Choosing the Right Metric

| Metric | Output | Best for |
|--------|--------|----------|
| **Levenshtein** | integer distance | General edit distance: insertions, deletions, substitutions |
| **Hamming** | integer distance | Position-by-position differences in aligned strings |
| **Damerau-Levenshtein** | integer distance | Edit distance when adjacent swaps should be cheap |
| **Jaro** | float similarity | Short strings where matching and transposition both matter |
| **Jaro-Winkler** | float similarity | Short strings where a shared prefix should help the score |
| **LCS** | integer length | Ordered overlap, diff-like comparisons, preserved subsequences |

**Key principle:** every function compares *grapheme++ clusters*, not raw Unicode code points. That is what makes the distances meaningful for Sinhala, Tamil, Arabic, emoji, and other multi-codepoint writing systems.
